In [1]:
import networkx as nx
import pandas as pd
from tqdm import tqdm
import reverse_geocoder as rg
# from utils.utils import town_to_planning_region
from utils.utils import town_to_historical_county
import pickle

In [2]:
tqdm.pandas()

## Load Yale Data

In [3]:
# Load the YCOM_2010_2024 sheet and filter for county rows with x2020 values

yale_2020 = pd.read_excel("YCOM_2024_publicdata.xlsx", sheet_name="YCOM_2010_2024")
print("Yale columns:", yale_2020.columns.tolist())

yale_2020_county = yale_2020[yale_2020["GeoType"] == "county"].copy()
yale_2020_county = yale_2020_county[yale_2020_county["x2020"].notna()].copy()
yale_2020_county = yale_2020_county[["GeoName", "GeoType", "x2020"]].copy()

def separate_county_and_state(row):
    split = row["GeoName"].split(",")
    county = split[0].strip()
    state = split[1].strip()
    return pd.Series([county, state])
    

yale_2020_county[["county", "state"]] = yale_2020_county.progress_apply(separate_county_and_state, axis=1)

yale_2020_county.head()

# Later, use yale_2020_county["x2020"] as your 2020 opinion value when matching by county/state.

Yale columns: ['GeoID', 'GeoName', 'GeoType', 'worried', 'x2010', 'x2011', 'x2012', 'x2013', 'x2014', 'x2015', 'x2016', 'x2017', 'x2018', 'x2019', 'x2020', 'x2021', 'x2022', 'x2023', 'x2024']


100%|██████████| 188520/188520 [00:10<00:00, 17441.39it/s]


,GeoName,GeoType,x2020,county,state
4523,"Autauga County, Alabama",county,48.88,Autauga County,Alabama
4524,"Baldwin County, Alabama",county,50.36,Baldwin County,Alabama
4525,"Barbour County, Alabama",county,56.91,Barbour County,Alabama
4526,"Bibb County, Alabama",county,48.19,Bibb County,Alabama
4527,"Blount County, Alabama",county,43.14,Blount County,Alabama


## Load Gowalla data

In [4]:
gowalla_total = pd.read_csv("gowalla_data/Gowalla_totalCheckins.txt",     
                             sep='\t', header=None)
gowalla_total.columns = ['userid','timestamp','latitude','longitude','spotid']
gowalla_total.head()

,userid,timestamp,latitude,longitude,spotid
0,0,2010-10-19T23:55:27Z,30.235909,-97.795140,22847
1,0,2010-10-18T22:17:43Z,30.269103,-97.749395,420315
2,0,2010-10-17T23:42:03Z,30.255731,-97.763386,316637
3,0,2010-10-17T19:26:05Z,30.263418,-97.757597,16516
4,0,2010-10-16T18:50:42Z,30.274292,-97.740523,5535878


## Get county for each entry in Gowalla data

In [5]:
coords = list(zip(gowalla_total['latitude'], gowalla_total['longitude']))

batch_size = 100000 
results = []

for i in tqdm(range(0, len(coords), batch_size)):
    chunk = coords[i:i+batch_size]
    res_chunk = rg.search(chunk, mode=2)
    results.extend(res_chunk)

  0%|          | 0/65 [00:00<?, ?it/s]

Loading formatted geocoded file...


100%|██████████| 65/65 [01:07<00:00,  1.04s/it]


In [6]:
gowalla_total.head()

,userid,timestamp,latitude,longitude,spotid
0,0,2010-10-19T23:55:27Z,30.235909,-97.795140,22847
1,0,2010-10-18T22:17:43Z,30.269103,-97.749395,420315
2,0,2010-10-17T23:42:03Z,30.255731,-97.763386,316637
3,0,2010-10-17T19:26:05Z,30.263418,-97.757597,16516
4,0,2010-10-16T18:50:42Z,30.274292,-97.740523,5535878


In [7]:
gowalla_total['country'] = [res['cc'] for res in results]
gowalla_total['state'] = [res['admin1'] for res in results]
gowalla_total['county'] = [res['admin2'] for res in results]
gowalla_total["city_name"] = [res["name"] for res in results]

In [8]:
gowalla_total.to_csv("gowalla_total.csv")

In [9]:
# gowalla_total = pd.read_csv("gowalla_total.csv")

In [10]:
gowalla_us = gowalla_total[gowalla_total["country"] == "US"]
gowalla_empty_state = gowalla_us[gowalla_us["state"] == ""]
assert len(gowalla_empty_state) == 0

In [11]:
us_users_checked_in = set(gowalla_us["userid"])
number_of_us_users = len(us_users_checked_in)
print(number_of_us_users)

53362


### Handle county edge cases to match Yale county format

In [12]:
def adjust_city_of_county(row):
    split = row.split("City of")
    city_name = split[1].strip()
    new_name = city_name + " City"
    return new_name

def adjust_saint_count(row):
    return row.replace("Saint ", "St. ")

In [13]:
# Keep CT county names from reverse geocoder; Yale 2020 expects actual counties, not planning regions.
connecticut_mask = gowalla_us["state"] == "Connecticut"
gowalla_us.loc[connecticut_mask, "county"] = gowalla_us.loc[connecticut_mask, "city_name"].map(town_to_historical_county)

# handle edge case for Washington D.C
washington_dc_mask = gowalla_us["city_name"] == "Washington, D.C."
gowalla_us.loc[washington_dc_mask, "county"] = "District of Columbia"
gowalla_us.loc[washington_dc_mask, "state"] = "District of Columbia"

# handle edge case for New York City
nyc_mask = ((gowalla_us["county"] == "")  | gowalla_us["county"].isna()) & (gowalla_us["city_name"] == "New York City")
gowalla_us.loc[nyc_mask, "county"] = "New York County"

## ex: City of Charlottesville is Charlottesville City in Yale Data
city_of_mask = gowalla_us['county'].str.startswith("City of")
gowalla_us.loc[city_of_mask, 'county'] = gowalla_us.loc[city_of_mask, 'county'].progress_apply(adjust_city_of_county)

# Saint -> St.
saint_mask = gowalla_us['county'].str.startswith("Saint")
gowalla_us.loc[saint_mask, 'county'] = gowalla_us.loc[saint_mask, 'county'].progress_apply(adjust_saint_count)

# ene mask
ene_mask = gowalla_us["county"] == "Dona Ana County"
gowalla_us.loc[ene_mask, "county"] = "Do\u00F1a Ana County"

# missing county (Bronx -> Bronx County)
missing_county_mask = gowalla_us["county"] == "Bronx"
gowalla_us.loc[missing_county_mask, "county"] = "Bronx County"

# desoto county
de_soto_mask = gowalla_us["county"] == "De Soto County"
gowalla_us.loc[de_soto_mask, "county"] = "DeSoto County"

bedford_mask = (gowalla_us["county"] == "Bedford City") & (gowalla_us["state"] == "Virginia")
gowalla_us.loc[bedford_mask, "county"] = "Bedford County"


100%|██████████| 34751/34751 [00:00<00:00, 1907929.29it/s]


In [14]:
assert len(gowalla_us[(gowalla_us["county"] == "")  | (gowalla_us["county"].isna())]) == 0 # should be empty
assert len(gowalla_us[(gowalla_us["state"] == "")  | (gowalla_us["state"].isna())]) == 0 # should be empty

### Set the county for each user by the most frequent location

In [15]:
gowalla_with_county_state_count = gowalla_us.groupby(['userid','county', 'state']).size().reset_index(name='Count')
max_counts = gowalla_with_county_state_count.groupby('userid')['Count'].transform('max')
filtered_gowalla = gowalla_with_county_state_count[gowalla_with_county_state_count['Count'] == max_counts]

In [16]:
assert len(set(filtered_gowalla["userid"])) == number_of_us_users # should be 53362
filtered_gowalla.to_csv("gowalla_with_county_state.csv")

# Filter edges to remove users that didn't check in

In [17]:
filtered_edges = []
with open("gowalla_data/Gowalla_edges.txt") as f:
    for i, line in enumerate(f):
        edge = line.strip().split()
        edge[0] = int(edge[0])
        edge[1] = int(edge[1])
        if (edge[0] in us_users_checked_in and edge[1] in us_users_checked_in):
            filtered_edges.append([edge[0], edge[1]])

with open("filtered_edge_list.txt", "w") as f:
    for u, v in filtered_edges:
        f.write(f"{u} {v}\n")

# Create network graph

In [18]:
G1 = nx.read_edgelist("filtered_edge_list.txt", create_using = nx.Graph(), nodetype=int)

In [19]:
mismatch = set()
for node in G1:
    matches = filtered_gowalla[filtered_gowalla['userid'] == node]
    if not matches.empty:
        user_county = matches["county"].iloc[0]
        user_state = matches["state"].iloc[0]

        G1.nodes[node]['county'] = user_county
        G1.nodes[node]['state'] = user_state

        yale_match = yale_2020_county[
            (yale_2020_county["state"] == user_state) & 
            (yale_2020_county["county"] == user_county)
        ]

        if yale_match.empty:
            print(f"Could not find county/state pair {user_county} - {user_state} in Yale 2020 dataset")
            continue

        net_opinion = yale_match["x2020"].iloc[0]

        G1.nodes[node]['baseline_opinion'] = net_opinion
        G1.nodes[node]['current_opinion'] = net_opinion
        G1.nodes[node]['susceptibility'] = 0.5

        for col in yale_2020_county.columns:
            if col not in ["state", "county"]:
                G1.nodes[node][col] = yale_match[col].iloc[0]


In [20]:
# Assign edge weights based on geographic interactions
intra_county_weight = 1.0  # Stronger influence if in the same county
inter_county_weight = 0.5  # Weaker influence if bridging different counties

for u, v in G1.edges():
    county_u = G1.nodes[u].get('county')
    county_v = G1.nodes[v].get('county')
    
    if county_u == county_v and county_u is not None:
        G1[u][v]['weight'] = intra_county_weight
    else:
        G1[u][v]['weight'] = inter_county_weight

# The FJ model usually requires the adjacency matrix W to be row-stochastic.
# We can normalize the weights for each node so their outgoing edge weights sum to 1.
for node in G1.nodes():
    total_weight = sum([data['weight'] for _, _, data in G1.edges(node, data=True)])
    if total_weight > 0:
        for neighbor in G1.neighbors(node):
            G1[node][neighbor]['normalized_weight'] = G1[node][neighbor]['weight'] / total_weight

In [21]:
with open('network_graph_fj_2020.pickle', 'wb') as f:
    pickle.dump(G1, f, pickle.HIGHEST_PROTOCOL)